In [2]:
import os
import numpy as np
import soundfile as sf
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import math
from tensorflow.keras import layers,models
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [6]:
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')
labels = np.load('/kaggle/input/time-domain/labels.npy')

In [7]:
max_value = np.max(np.abs(data))
data = data / max_value

In [8]:
df = pd.DataFrame(data[0])
df

,0,1,2,3,4,5,6,7
0,-0.000083,0.000956,0.000683,0.000084,0.003042,0.000090,0.000058,0.000029
1,-0.000507,-0.004144,0.001337,-0.000108,0.004349,0.000175,0.000140,0.000073
2,-0.000501,-0.003888,0.001519,-0.000061,0.005378,0.000199,0.000156,0.000078
3,0.000213,0.008767,0.001856,0.000588,0.010644,0.000248,0.000131,0.000056
4,0.001190,0.024331,0.001685,0.001339,0.014646,0.000233,0.000045,0.000002
...,...,...,...,...,...,...,...,...
15995,0.005224,0.166611,0.032341,0.010519,0.185818,0.004180,0.002293,0.000902
15996,0.034103,0.634627,0.029025,0.032863,0.310917,0.003946,-0.000194,-0.000612
15997,0.043036,0.728150,0.008568,0.035697,0.253617,0.001396,-0.002573,-0.001797
15998,0.026728,0.389247,-0.017163,0.017317,0.047019,-0.001972,-0.003461,-0.001966


In [9]:
data_2d = data.reshape(-1, 8)  # New shape: (402 * 16000, 8) = (6432000, 8)

# Expand labels: Repeat each label 16000 times to match data
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Convert to DataFrame
df = pd.DataFrame(data_2d, columns=[f'Sensor_{i+1}' for i in range(8)])
df['Label'] = labels_2d  # Assign labels

# Display DataFrame shape and sample
print(df.shape)  # Expected: (6432000, 9) → 8 sensor columns + 1 label column
print(df.head())


(6432000, 9)
   Sensor_1  Sensor_2  Sensor_3  Sensor_4  Sensor_5  Sensor_6  Sensor_7  \
0 -0.000083  0.000956  0.000683  0.000084  0.003042  0.000090  0.000058   
1 -0.000507 -0.004144  0.001337 -0.000108  0.004349  0.000175  0.000140   
2 -0.000501 -0.003888  0.001519 -0.000061  0.005378  0.000199  0.000156   
3  0.000213  0.008767  0.001856  0.000588  0.010644  0.000248  0.000131   
4  0.001190  0.024331  0.001685  0.001339  0.014646  0.000233  0.000045   

   Sensor_8  Label  
0  0.000029      0  
1  0.000073      0  
2  0.000078      0  
3  0.000056      0  
4  0.000002      0  


In [10]:
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_2d, test_size=0.2, random_state=42)

In [7]:
input_dim = data_2d.shape[1]  # 8 sensors
output_dim = len(np.unique(labels_2d))  # Number of unique classes

# Build the model
model = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),  # First hidden layer 128
    layers.Dropout(0.1),  # Dropout for regularization 0.1 all
    layers.Dense(64, activation='relu'),  # Second hidden layer 64
    layers.Dropout(0.1),
    layers.Dense(32, activation='relu'),  # Third hidden layer 32
    layers.Dropout(0.1),
    # layers.Dense(64, activation='relu'),  # Third hidden layer 32
    # layers.Dropout(0.15),
    # layers.Dense(32, activation='relu'),  # Third hidden layer 32
    # layers.Dropout(0.15),
    layers.Dense(output_dim, activation='softmax')  # Output layer for classification
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',  # Categorical classification loss
              metrics=['accuracy'])

# Show model summary
model.summary()


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           1,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,620 (45.39 KB)

 Trainable params: 11,620 (45.39 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

In [9]:
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
          callbacks=[early_stop, lr_reducer])

Epoch 1/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 273s 2ms/step - accuracy: 0.5105 - loss: 1.0426 - val_accuracy: 0.6626 - val_loss: 0.7566 - learning_rate: 0.0010
Epoch 2/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 272s 2ms/step - accuracy: 0.6328 - loss: 0.8150 - val_accuracy: 0.6831 - val_loss: 0.7120 - learning_rate: 0.0010
Epoch 3/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 272s 2ms/step - accuracy: 0.6534 - loss: 0.7753 - val_accuracy: 0.6938 - val_loss: 0.6917 - learning_rate: 0.0010
Epoch 4/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 272s 2ms/step - accuracy: 0.6655 - loss: 0.7503 - val_accuracy: 0.6583 - val_loss: 0.7611 - learning_rate: 0.0010
Epoch 5/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 270s 2ms/step - accuracy: 0.6749 - loss: 0.7302 - val_accuracy: 0.7021 - val_loss: 0.6756 - learning_rate: 0.0010
Epoch 6/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 269s 2ms/step - accuracy: 0.6826 - loss: 0.7155 - val_accuracy: 0.7090 - val_loss: 0.6565 - learning_rate: 0.0010
Epoch 7/50
160800/160800 ━━━━━━━━━━━━━━━

2nd Approach 

In [10]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Reshape data (Flatten time-series into features per sensor)
data_2d = data.reshape(-1, 8)  # New shape: (402 * 16000, 8) = (6432000, 8)

# Expand labels: Repeat each label 16000 times to match reshaped data
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Convert class labels to angles (assuming 4 classes equally distributed in 180°)
angles = labels_2d * (180 / 4)  # Class 0 → 0°, Class 1 → 45°, Class 2 → 90°, Class 3 → 135°

# Convert angles to sin and cos encoding
labels_sin = np.sin(np.radians(angles))
labels_cos = np.cos(np.radians(angles))

# Create final label array (2D regression targets)
labels_continuous = np.column_stack((labels_sin, labels_cos))

# Standardize sensor data
scaler = StandardScaler()
data_2d = scaler.fit_transform(data_2d)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_continuous, test_size=0.2, random_state=42)

# Model Hyperparameters
input_dim = X_train.shape[1]  # 8 sensors

# Build the model
model2 = tf.keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(2, activation='linear')  # Predict (sin, cos) values
])

# Custom Angular Loss Function
def angular_loss(y_true, y_pred):
    """ Loss function to minimize angular error. """
    true_sin, true_cos = tf.split(y_true, 2, axis=1)
    pred_sin, pred_cos = tf.split(y_pred, 2, axis=1)
    
    # Compute squared differences in sin & cos space
    loss = tf.keras.losses.MeanSquaredError()(true_sin, pred_sin) + \
           tf.keras.losses.MeanSquaredError()(true_cos, pred_cos)
    return loss

# Compile the model
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
model2.compile(optimizer=optimizer, loss=angular_loss, metrics=['mae'])  # Mean Absolute Error for monitoring

# Show model summary
model2.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

# Train model
model2.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
          callbacks=[early_stop, lr_reducer])

# ------------------- Prediction & Post-processing -------------------

# Predict on test data
pred_sin_cos = model2.predict(X_test)

# Convert back to angle
pred_angles = np.degrees(np.arctan2(pred_sin_cos[:, 0], pred_sin_cos[:, 1]))  # atan2(sin, cos) → Angle

# Convert back to class labels
predicted_classes = np.round(pred_angles / (180 / 4)) % 4  # Convert back to class indices

# Display sample predictions
print("Predicted Angles:", pred_angles[:10])
print("Predicted Classes:", predicted_classes[:10])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 256)                 │           2,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 2)                   │              66 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,522 (185.63 KB)

 Trainable params: 46,562 (181.88 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 346s 2ms/step - loss: 0.2829 - mae: 0.2826 - val_loss: 0.1719 - val_mae: 0.2133 - learning_rate: 0.0010
Epoch 2/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 336s 2ms/step - loss: 0.2116 - mae: 0.2429 - val_loss: 0.1569 - val_mae: 0.2046 - learning_rate: 0.0010
Epoch 3/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 335s 2ms/step - loss: 0.2044 - mae: 0.2375 - val_loss: 0.1566 - val_mae: 0.2019 - learning_rate: 0.0010
Epoch 4/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 340s 2ms/step - loss: 0.2005 - mae: 0.2346 - val_loss: 0.1526 - val_mae: 0.2064 - learning_rate: 0.0010
Epoch 5/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 331s 2ms/step - loss: 0.1994 - mae: 0.2338 - val_loss: 0.1333 - val_mae: 0.1888 - learning_rate: 0.0010
Epoch 6/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 333s 2ms/step - loss: 0.1978 - mae: 0.2327 - val_loss: 0.1636 - val_mae: 0.2042 - learning_rate: 0.0010
Epoch 7/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 332s 2ms/step - loss: 0.1969 - mae: 0.2320 - val_loss:

###3rd approach

In [11]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Reshape data (Flatten time-series into features per sensor)
data_2d = data.reshape(-1, 8)  # New shape: (402 * 16000, 8) = (6432000, 8)

# Expand labels: Repeat each label 16000 times to match reshaped data
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Standardize sensor data
scaler = StandardScaler()
data_2d = scaler.fit_transform(data_2d)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_2d, test_size=0.2, random_state=42)

# Model Hyperparameters
input_dim = X_train.shape[1]  # 8 sensors
output_dim = len(np.unique(labels_2d))  # Number of unique classes (4)

# Build the model
model3 = tf.keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(output_dim, activation='softmax')  # Output: 4 classes
])

# Compile the model
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
model3.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Show model summary
model3.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

# Train model
model3.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
          callbacks=[early_stop, lr_reducer])

# ------------------- Prediction & Post-processing -------------------

# Predict on test data
predictions = model3.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)  # Convert softmax output to class indices

# Display sample predictions
print("Predicted Classes:", predicted_classes[:10])
print("True Classes:", y_test[:10])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                      │ (None, 256)                 │           2,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_8 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_9 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,588 (185.89 KB)

 Trainable params: 46,628 (182.14 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 344s 2ms/step - accuracy: 0.5953 - loss: 0.9041 - val_accuracy: 0.7246 - val_loss: 0.6539 - learning_rate: 0.0010
Epoch 2/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 337s 2ms/step - accuracy: 0.6572 - loss: 0.7900 - val_accuracy: 0.7268 - val_loss: 0.6401 - learning_rate: 0.0010
Epoch 3/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 336s 2ms/step - accuracy: 0.6681 - loss: 0.7689 - val_accuracy: 0.7240 - val_loss: 0.6447 - learning_rate: 0.0010
Epoch 4/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 338s 2ms/step - accuracy: 0.6769 - loss: 0.7528 - val_accuracy: 0.7271 - val_loss: 0.6423 - learning_rate: 0.0010
Epoch 5/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 335s 2ms/step - accuracy: 0.6863 - loss: 0.7328 - val_accuracy: 0.7438 - val_loss: 0.6086 - learning_rate: 5.0000e-04
Epoch 6/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 336s 2ms/step - accuracy: 0.6890 - loss: 0.7259 - val_accuracy: 0.7298 - val_loss: 0.6383 - learning_rate: 5.0000e-04
Epoch 7/50
160800/160800 ━━━━━━━

**4th Approach**

In [12]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Reshape data (Flatten time-series into features per sensor)
data_2d = data.reshape(-1, 8)  # New shape: (402 * 16000, 8) = (6432000, 8)

# Expand labels: Repeat each label 16000 times to match reshaped data
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Standardize features
scaler = StandardScaler()
data_2d = scaler.fit_transform(data_2d)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_2d, test_size=0.2, stratify=labels_2d, random_state=42)

# Model Hyperparameters
input_dim = X_train.shape[1]  # 8 sensors
output_dim = len(np.unique(labels_2d))  # Number of unique classes

# Build the model
model4 = tf.keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(output_dim, activation='softmax')  # Output layer for classification
])

# Compile the model with AdamW optimizer (better weight decay handling)
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
model4.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Show model summary
model4.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

# Train model
model4.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
          callbacks=[early_stop, lr_reducer])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                     │ (None, 256)                 │           2,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_9                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_14 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_18 (Dense)                     │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,588 (185.89 KB)

 Trainable params: 46,628 (182.14 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 346s 2ms/step - accuracy: 0.5984 - loss: 0.8970 - val_accuracy: 0.7059 - val_loss: 0.6842 - learning_rate: 0.0010
Epoch 2/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 343s 2ms/step - accuracy: 0.6563 - loss: 0.7901 - val_accuracy: 0.7169 - val_loss: 0.6627 - learning_rate: 0.0010
Epoch 3/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 344s 2ms/step - accuracy: 0.6679 - loss: 0.7683 - val_accuracy: 0.7264 - val_loss: 0.6487 - learning_rate: 0.0010
Epoch 4/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 344s 2ms/step - accuracy: 0.6742 - loss: 0.7564 - val_accuracy: 0.7074 - val_loss: 0.6773 - learning_rate: 0.0010
Epoch 5/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 344s 2ms/step - accuracy: 0.6748 - loss: 0.7553 - val_accuracy: 0.7350 - val_loss: 0.6310 - learning_rate: 0.0010
Epoch 6/50
160800/160800 ━━━━━━━━━━━━━━━━━━━━ 338s 2ms/step - accuracy: 0.6811 - loss: 0.7439 - val_accuracy: 0.7264 - val_loss: 0.6392 - learning_rate: 0.0010
Epoch 7/50
160800/160800 ━━━━━━━━━━━━━━━

In [13]:
# Predict on test data
predictions = model4.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)  # Convert softmax output to class indices

# Display sample predictions
print("Predicted Classes:", predicted_classes[:10])
print("True Classes:", y_test[:10])

40200/40200 ━━━━━━━━━━━━━━━━━━━━ 50s 1ms/step
Predicted Classes: [1 1 1 2 0 2 3 0 3 2]
True Classes: [0 1 1 2 0 1 3 1 3 2]


In [3]:
x_t=np.array([-0.000083,	0.000956,	0.000683,	0.000084,	0.003042,	0.000090,	0.000058,	0.000029])
x_t

array([-8.300e-05,  9.560e-04,  6.830e-04,  8.400e-05,  3.042e-03,
        9.000e-05,  5.800e-05,  2.900e-05])

In [4]:
print("hello")

hello


**CNN Approach**

In [14]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Standardize each sensor's data
scaler = StandardScaler()
for i in range(data.shape[-1]):  # Normalize across each sensor independently
    data[:, :, i] = scaler.fit_transform(data[:, :, i])

# Reshape data for CNN
X = data  # Shape: (402, 16000, 8) → CNN expects (samples, time_steps, features)
y = labels  # Shape: (402,)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Model Hyperparameters
time_steps = X_train.shape[1]  # 16000 samples per instance
num_sensors = X_train.shape[2]  # 8 sensors
output_dim = len(np.unique(y_train))  # Number of classes

# Build CNN Model
model_cnn = tf.keras.Sequential([
    layers.Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(time_steps, num_sensors)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(filters=128, kernel_size=5, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(filters=256, kernel_size=5, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(output_dim, activation='softmax')  # Classification Output
])

# Compile the Model
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
model_cnn.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Show Model Summary
model_cnn.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

# Train Model
history = model_cnn.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
                         callbacks=[early_stop, lr_reducer])



/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)                    │ (None, 15996, 64)           │           2,624 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 15996, 64)           │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_6 (MaxPooling1D)       │ (None, 7998, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_7 (Conv1D)                    │ (None, 7994, 128)           │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 7994, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_7 (MaxPooling1D)       │ (None, 3997, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_8 (Conv1D)                    │ (None, 3993, 256)           │         164,096 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 3993, 256)           │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_8 (MaxPooling1D)       │ (None, 1996, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 510976)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 128)                 │      65,405,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 4)                   │             260 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 65,623,940 (250.34 MB)

 Trainable params: 65,622,660 (250.33 MB)

 Non-trainable params: 1,280 (5.00 KB)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.3017 - loss: 1.9204 - val_accuracy: 0.2593 - val_loss: 5.3406 - learning_rate: 0.0010
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 161ms/step - accuracy: 0.3869 - loss: 1.5434 - val_accuracy: 0.2469 - val_loss: 3.8264 - learning_rate: 0.0010
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 161ms/step - accuracy: 0.5000 - loss: 1.2669 - val_accuracy: 0.2469 - val_loss: 2.3420 - learning_rate: 0.0010
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 163ms/step - accuracy: 0.5959 - loss: 0.9150 - val_accuracy: 0.3951 - val_loss: 1.3479 - learning_rate: 0.0010
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.6050 - loss: 0.8507 - val_accuracy: 0.2963 - val_loss: 1.6941 - learning_rate: 0.0010
Epoch 6/50
10/11 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.6163 - loss: 0.8685
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.6216 - loss: 0.8557 - val

In [17]:
model_cnn.save('/kaggle/working/cnn_sensor_model.h5')


In [18]:
from IPython.display import FileLink
FileLink('/kaggle/working/cnn_sensor_model.h5')


/kaggle/working/cnn_sensor_model.h5

In [19]:
from tensorflow.keras.models import load_model
model_cnn = load_model('/kaggle/input/your-dataset-folder/cnn_sensor_model.h5')


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/kaggle/input/your-dataset-folder/cnn_sensor_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

**HyperParameter Tuning**

In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna
from keras_tuner import RandomSearch, BayesianOptimization
from tensorflow.keras import layers

# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Reshape data (Flatten time-series into features per sensor)
data_2d = data.reshape(-1, 8)  # Shape: (402 * 16000, 8)
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Standardize features
scaler = StandardScaler()
data_2d = scaler.fit_transform(data_2d)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_2d, test_size=0.2, stratify=labels_2d, random_state=42)

### MLP MODEL WITH OPTUNA ###
def objective(trial):
    model = tf.keras.Sequential([
        layers.Dense(trial.suggest_int('units1', 64, 512, step=64), activation='relu', input_shape=(8,)),
        layers.Dropout(trial.suggest_float('dropout1', 0.1, 0.5)),
        layers.Dense(trial.suggest_int('units2', 64, 256, step=64), activation='relu'),
        layers.Dropout(trial.suggest_float('dropout2', 0.1, 0.5)),
        layers.Dense(len(np.unique(labels)), activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=trial.suggest_loguniform('lr', 1e-4, 1e-2)),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=0)
    return max(history.history['val_accuracy'])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print("Best parameters:", study.best_params)

### CNN MODEL WITH KERAS TUNER ###
def build_cnn(hp):
    model = tf.keras.Sequential([
        layers.Conv1D(filters=hp.Choice('filters', [64, 128, 256]), kernel_size=hp.Choice('kernel_size', [3, 5]),
                      activation='relu', input_shape=(8, 1)),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)),
        layers.Dense(len(np.unique(labels)), activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('lr', [1e-2, 1e-3, 1e-4])),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

tuner = RandomSearch(build_cnn, objective='val_accuracy', max_trials=10)
tuner.search(X_train.reshape(-1, 8, 1), y_train, epochs=10, validation_split=0.2)
print("Best CNN parameters:", tuner.get_best_hyperparameters()[0].values)

### LSTM MODEL WITH OPTUNA ###
def lstm_objective(trial):
    model = tf.keras.Sequential([
        layers.LSTM(trial.suggest_int('lstm_units', 64, 512, step=64), return_sequences=True, input_shape=(8, 1)),
        layers.LSTM(trial.suggest_int('lstm_units2', 32, 256, step=32)),
        layers.Dropout(trial.suggest_float('dropout', 0.1, 0.5)),
        layers.Dense(len(np.unique(labels)), activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=trial.suggest_loguniform('lr', 1e-4, 1e-2)),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train.reshape(-1, 8, 1), y_train, epochs=10, validation_data=(X_test.reshape(-1, 8, 1), y_test), verbose=0)
    return max(history.history['val_accuracy'])

study = optuna.create_study(direction='maximize')
study.optimize(lstm_objective, n_trials=10)
print("Best LSTM parameters:", study.best_params)

### TRANSFORMER MODEL WITH BAYESIAN OPTIMIZATION ###
def build_transformer(hp):
    input_layer = layers.Input(shape=(8, 1))
    x = layers.MultiHeadAttention(num_heads=hp.Choice('heads', [4, 8]), key_dim=64)(input_layer, input_layer)
    x = layers.Dense(hp.Int('dense_units', 128, 512, step=128), activation='relu')(x)
    x = layers.Dense(len(np.unique(labels)), activation='softmax')(x)
    model = tf.keras.Model(inputs=input_layer, outputs=x)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('lr', [1e-4, 1e-5])),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

tuner = BayesianOptimization(build_transformer, objective='val_accuracy', max_trials=10)
tuner.search(X_train.reshape(-1, 8, 1), y_train, epochs=10, validation_split=0.2)
print("Best Transformer parameters:", tuner.get_best_hyperparameters()[0].values)


[I 2025-03-07 06:05:37,189] A new study created in memory with name: no-name-c14e2aca-7a3f-4f61-bdc5-c5a0778449e6
/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
<ipython-input-1-7a8a34a0565c>:37: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=trial.suggest_loguniform('lr', 1e-4, 1e-2)),
[W 2025-03-07 06:10:15,128] Trial 0 failed with parameters: {'units1': 192, 'dropout1': 0.26621921314526176, 'units2': 128, 'dropout2': 0.17898508481491004, 'lr': 0.0007257190658320027} because of the foll

KeyboardInterrupt: 

**Hypertuning 2**

In [2]:
# Hyperparameter Tuning for ML Models (MLP, CNN, LSTM, Transformer)

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna  # For hyperparameter tuning
from keras_tuner import RandomSearch
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau




In [3]:
# Load Data
data = np.load('/kaggle/input/time-domain/time_domain_data.npy')  # Shape: (402, 16000, 8)
labels = np.load('/kaggle/input/time-domain/labels.npy')  # Shape: (402,)

# Normalize Data
data /= np.max(np.abs(data))  # Scale between -1 and 1

# Reshape data (Flatten time-series into features per sensor)
data_2d = data.reshape(-1, 8)  # New shape: (402 * 16000, 8) = (6432000, 8)

# Expand labels: Repeat each label 16000 times to match reshaped data
labels_2d = np.repeat(labels, 16000)  # Shape: (6432000,)

# Standardize features
scaler = StandardScaler()
data_2d = scaler.fit_transform(data_2d)

In [4]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data_2d, labels_2d, test_size=0.2, stratify=labels_2d, random_state=42)


In [ ]:
# Define MLP Model for Optuna Tuning
def create_mlp(trial):
    model = tf.keras.Sequential()
    model.add(layers.Dense(trial.suggest_int('units1', 128, 512), activation='relu', input_shape=(8,)))
    model.add(layers.Dropout(trial.suggest_float('dropout1', 0.1, 0.5)))
    model.add(layers.Dense(trial.suggest_int('units2', 64, 256), activation='relu'))
    model.add(layers.Dropout(trial.suggest_float('dropout2', 0.1, 0.5)))
    model.add(layers.Dense(len(np.unique(y_train)), activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=trial.suggest_float('lr', 1e-5, 1e-2, log=True)),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Hyperparameter tuning with Optuna
def objective(trial):
    model = create_mlp(trial)
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=32, verbose=0)
    return history.history['val_accuracy'][-1]

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)



# Best Model Training
best_params = study.best_params
best_model = create_mlp(study.best_trial)

# Show Model Summary
best_model.summary()

best_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)

# Save the model
best_model.save('/kaggle/working/mlp_model.h5')

[I 2025-03-07 06:26:23,936] A new study created in memory with name: no-name-b191ae3f-b580-4a34-af52-7606d4c334ca
/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2025-03-07 07:05:08,446] Trial 0 finished with value: 0.2748258709907532 and parameters: {'units1': 508, 'dropout1': 0.16634422259492193, 'units2': 190, 'dropout2': 0.32884189795084473, 'lr': 0.005568874213654941}. Best is trial 0 with value: 0.2748258709907532.
[I 2025-03-07 07:44:03,167] Trial 1 finished with value: 0.800438404083252 and parameters: {'units1': 438, 'dropout1': 0.20322787710998347, 'units2': 249, 'dropout2': 0.44764045714729084, 'lr': 1.0631413815342016e-05}. Best is trial 1 with value: 0.800438404083252.
[I 2025-03-07 08:23:09,

In [ ]:
# Compile the Model
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
model_cnn.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Show Model Summary
model_cnn.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

# Train Model
history = model_cnn.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test),
                         callbacks=[early_stop, lr_reducer])

**HP + CNN**

In [ ]:
# CNN Model with Keras Tuner
def build_cnn(hp):
    model = tf.keras.Sequential()
    model.add(layers.Conv1D(hp.Int('conv1_filters', 32, 128, step=32), kernel_size=3, activation='relu', input_shape=(8, 1)))
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Conv1D(hp.Int('conv2_filters', 32, 128, step=32), kernel_size=3, activation='relu'))
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Flatten())
    model.add(layers.Dense(hp.Int('dense_units', 64, 256, step=64), activation='relu'))
    model.add(layers.Dense(len(np.unique(y_train)), activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 1e-5, 1e-2, sampling='LOG')),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Hyperparameter tuning for CNN
tuner = RandomSearch(build_cnn, objective='val_accuracy', max_trials=5, executions_per_trial=1, directory='/kaggle/working', project_name='cnn_tuning')
X_train_cnn = X_train.reshape(-1, 8, 1)
X_test_cnn = X_test.reshape(-1, 8, 1)
tuner.search(X_train_cnn, y_train, validation_data=(X_test_cnn, y_test), epochs=10, batch_size=32)

# Train best CNN model
best_cnn_model = tuner.get_best_models(num_models=1)[0]
# Show Model Summary
best_cnn_model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)


history4 = best_cnn_model.fit(X_train_cnn, y_train, validation_data=(X_test_cnn, y_test), epochs=50, batch_size=32,callbacks=[early_stop, lr_reducer)
best_cnn_model.save('/kaggle/working/cnn_model.h5')

**HP+LSTM**

In [ ]:
# LSTM Model
def create_lstm(trial):
    model = tf.keras.Sequential()
    model.add(layers.LSTM(trial.suggest_int('lstm_units', 32, 128), input_shape=(8, 1)))
    model.add(layers.Dense(trial.suggest_int('dense_units', 64, 256), activation='relu'))
    model.add(layers.Dense(len(np.unique(y_train)), activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(trial.suggest_float('lr', 1e-5, 1e-2, log=True)),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Train best LSTM Model
best_lstm_model = create_lstm(study.best_trial)
best_lstm_model.fit(X_train_cnn, y_train, validation_data=(X_test_cnn, y_test), epochs=50, batch_size=32)
best_lstm_model.save('/kaggle/working/lstm_model.h5')


In [ ]:
# Transformer Model
class TransformerModel(tf.keras.Model):
    def __init__(self, num_heads, d_model, num_layers, dff, output_dim):
        super(TransformerModel, self).__init__()
        self.embedding = layers.Dense(d_model, activation='relu')
        self.transformer_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model) for _ in range(num_layers)]
        self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]
        self.output_layer = layers.Dense(output_dim, activation='softmax')
    
    def call(self, inputs):
        x = self.embedding(inputs)
        for mha, dense in zip(self.transformer_layers, self.dense_layers):
            x = mha(x, x)
            x = dense(x)
        return self.output_layer(x)

transformer_model = TransformerModel(num_heads=4, d_model=128, num_layers=2, dff=256, output_dim=len(np.unique(y_train)))
transformer_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
transformer_model.fit(X_train_cnn, y_train, validation_data=(X_test_cnn, y_test), epochs=50, batch_size=32)
transformer_model.save('/kaggle/working/transformer_model.h5')
